In [1]:
import os
import pandas as pd
import numpy as np
import ast 
import re, torch
from typing import List
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [2]:
SENTINEL = "<extra_id_0>"
PREFIX   = "predict_if_condition: "

In [3]:
def _clean(text: str) -> str:
    # Remove any leftover sentinels and tidy whitespace
    for i in range(100):
        text = text.replace(f"<extra_id_{i}>", "")
    text = re.sub(r"\s+", " ", text.strip())
    if text.endswith(":"):
        text = text[:-1].rstrip()
    return text

def load_model(model_dir: str):
    # Always load a T5 SentencePiece tokenizer
    tok = T5Tokenizer.from_pretrained(model_dir, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

    # Make sure generation IDs are set (some checkpoints lose these)
    if model.config.pad_token_id is None and tok.pad_token_id is not None:
        model.config.pad_token_id = tok.pad_token_id
    if model.config.eos_token_id is None and tok.eos_token_id is not None:
        model.config.eos_token_id = tok.eos_token_id
    if model.config.decoder_start_token_id is None:
        model.config.decoder_start_token_id = model.config.pad_token_id

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    return tok, model, device

@torch.no_grad()
def predict_condition(model_dir: str,
                      code_snippet: str,
                      max_new_tokens: int = 64) -> str:
    tok, model, device = load_model(model_dir)

    # Replace <mask> with the sentinel you trained with and add the task prefix
    code = code_snippet.replace("<mask>", SENTINEL)
    src  = PREFIX + code

    batch = tok(src, return_tensors="pt", truncation=True, max_length=1024)
    # T5 doesn't use token_type_ids
    batch.pop("token_type_ids", None)
    batch = {k: v.to(device) for k, v in batch.items()}

    out = model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        num_beams=1,
        early_stopping=True,
        decoder_start_token_id=model.config.decoder_start_token_id,
        eos_token_id=model.config.eos_token_id,
        pad_token_id=model.config.pad_token_id,
        length_penalty=0.0,
    )
    return _clean(tok.decode(out[0], skip_special_tokens=True))

@torch.no_grad()
def predict_many(model_dir: str, code_snippets: List[str], **gen_kwargs) -> List[str]:
    return [predict_condition(model_dir, s, **gen_kwargs) for s in code_snippets]


def post_process_condition(text: str, max_chars: int = 500) -> str:
    """
    Simple, robust cleaner for predicted conditions.
    - remove sentinels
    - cut at newline or trailing colon
    - normalize a few artifacts
    - collapse generic repetitions (words, parentheses)
    - dedupe repeated tail n-grams (e.g., `_type == "negative"` over and over)
    """
    if not isinstance(text, str):
        return ""

    s = text

    # 1) remove T5 sentinels
    s = re.sub(r"<extra_id_\d+>", "", s)

    # 2) cut at newline or trailing colon
    s = re.split(r"[\r\n]+|:\s*$", s, maxsplit=1)[0]

    # 3) normalize common artifacts / casings
    s = re.sub(r"\bnone[_\s]*type\b", "NoneType", s, flags=re.IGNORECASE)
    s = re.sub(r"\btrue\b", "True", s, flags=re.IGNORECASE)
    s = re.sub(r"\bfalse\b", "False", s, flags=re.IGNORECASE)
    s = re.sub(r"\bnone\b", "None", s, flags=re.IGNORECASE)

    # 4) collapse obvious repeats
    s = re.sub(r"(\b\w+\b)(?:\s*\1){2,}\b", r"\1", s)          # word word word -> word
    s = re.sub(r"(\([^()]*\))(?:\s*\1){2,}", r"\1", s)         # (x)(x)(x) -> (x)
    s = re.sub(r"(?:_?type){2,}\b", "type", s, flags=re.IGNORECASE)  # _type_type... -> type

    # 5) generic tail n-gram dedupe (handles `_type == "negative"` * N)
    def dedupe_tail_ngrams(t: str, max_n: int = 5) -> str:
        toks = re.findall(r'\w+|[^\s\w]', t)  # simple tokenization: words or single punct
        changed = True
        while changed:
            changed = False
            L = len(toks)
            for n in range(min(max_n, L // 2), 0, -1):
                # check if the last 2*n tokens are two identical n-grams
                if L >= 2 * n and toks[L-2*n:L-n] == toks[L-n:L]:
                    # remove one repetition
                    toks = toks[:L-n]
                    changed = True
                    break
        return "".join(
            [tok if re.fullmatch(r'[^\w\s]', tok) else (" " + tok) for tok in toks]
        ).strip()

    s = dedupe_tail_ngrams(s, max_n=5)

    # 6) collapse spaces and cap length
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_chars:
        s = s[:max_chars].rstrip()

    return s

In [12]:
model_dir = "t5_if_finetuned_backup"

# Quick demo
code_snippet = """\
    def check_function(x):
        if x % 2 == 0:
            return "even"
        else:
            return "odd"
    """

pred = predict_condition(model_dir, code_snippet)
pred = post_process_condition(pred)

print("Predicted if condition:", pred)
print(f"Number of predicted characters: {len(pred)}")

Predicted if condition: x% 2== 0 in[" even"," even"] in x"]
Number of predicted characters: 35


# 2. Evaluate and compare accuracy

In [5]:
# Load training data for evaluation
PATH_FILE_DATA = os.path.join(os.getcwd(), "dataset", "processed", "processed_data.csv")

df = pd.read_csv(PATH_FILE_DATA)
list_python_function = df["method_code"].tolist()
print(f"Number of functions to evaluate: {len(list_python_function)}")

Number of functions to evaluate: 1371223


In [6]:
def _mask_span_by_pos(code: str, sl: int, sc: int, el: int, ec: int, sentinel=SENTINEL) -> str:
    lines = code.splitlines(keepends=True)
    before = "".join(lines[:sl-1]) + lines[sl-1][:sc]
    after  = lines[el-1][ec:] + "".join(lines[el:])
    return before + sentinel + after

def extract_mask_all_ifs(code: str):
    """
    Yield dicts for EVERY ast.If (incl. elif/nested) with:
      - masked_code: the function with that single test replaced by SENTINEL
      - gold_cond:   exact source of the test (target)
      - idx:         0-based index of the if in source order
    """
    out = []
    code = code.strip("\n")
    if not code:
        return out
    try:
        tree = ast.parse(code)
    except Exception:
        return out

    if_nodes = [n for n in ast.walk(tree) if isinstance(n, ast.If)]
    if_nodes.sort(key=lambda n: (getattr(n, "lineno", 10**9), getattr(n, "col_offset", 10**9)))

    for i, n in enumerate(if_nodes):
        if not (hasattr(n.test, "lineno") and hasattr(n.test, "end_lineno")):
            continue
        # get exact condition text
        cond_text = None
        try:
            cond_text = ast.get_source_segment(code, n.test)
        except Exception:
            try:
                cond_text = ast.unparse(n.test)
            except Exception:
                cond_text = None
        if not cond_text:
            continue
        cond_text = cond_text.strip()
        if not cond_text:
            continue

        masked = _mask_span_by_pos(code, n.test.lineno, n.test.col_offset,
                                        n.test.end_lineno, n.test.end_col_offset,
                                        sentinel=SENTINEL)
        out.append({"idx": i, "masked_code": masked, "gold_cond": cond_text})
    return out

def _normalize(s: str) -> str:
    # light normalization for fair exact-match comparison
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\(\s*", "(", s)
    s = re.sub(r"\s*\)", ")", s)
    if s.endswith(":"):
        s = s[:-1].rstrip()
    return s


def code_tokenize(s: str):
    """Tokenize a Python-like condition into words and symbols."""
    _CODE_TOK = re.compile(r'\w+|[^\s\w]')
    return _CODE_TOK.findall(s or "")

# ---- BLEU helper ----
def nltk_sentence_bleu(pred: str, true_if: str) -> float:
    """
    Compute BLEU (0–100) between predicted and true condition strings.
    Uses short-sequence smoothing.
    """
    hyp = code_tokenize(pred)
    ref = code_tokenize(true_if)
    smoothie = SmoothingFunction().method3  
    return 100.0 * sentence_bleu([ref], hyp, smoothing_function=smoothie)

# --- main evaluation pipeline ---

def evaluate_function(model_dir: str, func_src: str, *, max_new_tokens=64, num_beams=4):
    """
    For a single function string:
      - create one masked sample per `if`,
      - run predict_condition(...) for each,
      - compare normalized exact match,
    Returns: pandas DataFrame with one row per `if` and summary printed.
    """
    # you already have this function defined elsewhere:
    # pred = predict_condition(model_dir, code_snippet)
    # we’ll just call it below.

    items = extract_mask_all_ifs(func_src)
    rows = []
    for it in items:
        # form the model input with the task prefix and sentinel already in the code
        src = f"{PREFIX}{it['masked_code']}"
        
        pred = predict_condition(model_dir, src)  
        pred = post_process_condition(pred)

        true_norm = _normalize(it["gold_cond"])
        pred_norm = _normalize(pred)
        
        bleu_score = nltk_sentence_bleu(pred, true_norm)
        
        rows.append({
            "if_index": it["idx"],
            "true_norm": true_norm,
            "pred_norm": pred_norm,
            "exact_match": true_norm == pred_norm,
            "bleu_score": bleu_score,
        })

    df = pd.DataFrame(rows, columns=[
        "if_index", "true_norm", "pred_norm", "exact_match", 'bleu_score'
    ])

    # tiny summary
    total = len(df)
    em = int(df["exact_match"].sum()) if total else 0
    acc = (em / total) if total else 0.0
    print(f"[evaluate_function] if_count={total}  exact_match={em}/{total} ({acc:.2%})")
    return df

# --- optional: evaluate a list of functions and aggregate ---

def evaluate_functions(model_dir: str, functions, *, max_new_tokens=64, num_beams=4):
    all_rows = []
    for fi, fsrc in enumerate(functions):
        df = evaluate_function(model_dir, fsrc, max_new_tokens=max_new_tokens, num_beams=num_beams)
        if len(df):
            df.insert(0, "function_index", fi)
            all_rows.append(df)
    if not all_rows:
        print("[evaluate_functions] No if-conditions found.")
        return pd.DataFrame()
    big = pd.concat(all_rows, ignore_index=True)
    overall = big["exact_match"].mean() if len(big) else 0.0
    print(f"[evaluate_functions] samples={len(big)}  overall_exact_match={overall:.2%}")
    return big


In [7]:
while 1:
    idx = np.random.randint(0, len(list_python_function))
    func = list_python_function[idx]
    
    if ("if " in func) or (" if" in func):
        break
    
print(func)

def _while_op_lowering_rule(
    ctx: LoweringContext, while_op: scf.WhileOp
) -> MlirLoweringRuleResult:
  if not inference_utils.should_have_layout(while_op):
    return _traverse_op_lowering_rule(ctx, while_op)

  before_block = while_op.before.blocks[0]
  after_block = while_op.after.blocks[0]
  condition_op = before_block.operations[len(before_block.operations) - 1]
  yield_op = after_block.operations[len(after_block.operations) - 1]

  in_layouts = (
      inference_utils.in_layouts(while_op)
      if inference_utils.should_have_in_layout(while_op)
      else []
  )
  out_layouts = (
      inference_utils.out_layouts(while_op)
      if inference_utils.should_have_out_layout(while_op)
      else []
  )

  if in_layouts:
    yield_layouts = inference_utils.in_layouts(yield_op)
    if in_layouts != yield_layouts:
      raise ValueError(
          f"Input layouts {in_layouts} do not match yield layouts"
          f" {yield_layouts}"
      )

  if out_layouts:
    condition_layouts = 

In [8]:
model_dir = "t5_if_finetuned_backup"

df = evaluate_function(model_dir, func, max_new_tokens=32, num_beams=4)
df.head()

[evaluate_function] if_count=5  exact_match=1/5 (20.00%)


,if_index,true_norm,pred_norm,exact_match,bleu_score
0,0,not inference_utils.should_have_layout(while_op),not ctx. is_traverse_op_lowering_rule(ctx. bef...,False,4.016138
1,1,in_layouts,in_layouts,True,35.355339
2,2,in_layouts != yield_layouts,in_layouts!= layouts,False,59.460356
3,3,out_layouts,in_layouts_op_op_op_op_op_op_op_op_op_op_op_op...,False,0.000000
4,4,out_layouts != condition_layouts,condition_layouts!= condition_layouts not None,False,30.213754
